# Lesson 4a: Temporal Difference Learning - Theory

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Understand TD Learning** - Bootstrap + sampling for online learning
2. **Master TD(0) Prediction** - One-step temporal difference
3. **Compare TD vs MC vs DP** - Bias-variance tradeoffs
4. **Learn SARSA** - On-policy TD control
5. **Learn Q-Learning** - Off-policy TD control
6. **Understand Convergence** - Theoretical guarantees
7. **Explore n-Step Methods** - Bridging TD and MC

Temporal Difference learning is one of the most important ideas in reinforcement learning, combining the best of Monte Carlo and Dynamic Programming.

---

## Table of Contents

1. [Introduction to TD Learning](#intro)
2. [TD(0) Prediction](#td-prediction)
3. [TD vs MC vs DP](#comparison)
4. [SARSA: On-Policy TD Control](#sarsa)
5. [Q-Learning: Off-Policy TD Control](#q-learning)
6. [Expected SARSA](#expected-sarsa)
7. [Convergence Theory](#convergence)
8. [n-Step TD Methods](#n-step)
9. [Maximization Bias](#max-bias)

---

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

np.random.seed(42)
print("✅ Setup complete")

## 1. Introduction to TD Learning <a id='intro'></a>

### The Central Idea

**Temporal Difference (TD) learning** is a method that combines:
- **Sampling** (like Monte Carlo): Learn from experience without a model
- **Bootstrapping** (like Dynamic Programming): Update estimates based on other estimates

### The TD Update Rule

**Monte Carlo** waits for actual return:
$$V(S_t) \leftarrow V(S_t) + \alpha[G_t - V(S_t)]$$

where $G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ...$

**TD(0)** uses estimated return:
$$V(S_t) \leftarrow V(S_t) + \alpha[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)]$$

**Key difference**: $R_{t+1} + \gamma V(S_{t+1})$ is an **estimate** of $G_t$

### TD Target and TD Error

**TD Target**:
$$R_{t+1} + \gamma V(S_{t+1})$$

**TD Error** (also called *temporal difference*):
$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

**TD Update** (compact form):
$$V(S_t) \leftarrow V(S_t) + \alpha \delta_t$$

### Why "Temporal Difference"?

The method is named because it uses the **difference between estimates at successive time steps**:

$$\delta_t = \underbrace{R_{t+1} + \gamma V(S_{t+1})}_{\text{estimate at } t+1} - \underbrace{V(S_t)}_{\text{estimate at } t}$$

This "difference across time" drives learning.

### Advantages of TD Learning

1. **Online Learning**: Update immediately after each step
   - Don't wait for episode end
   - Works for continuing (non-episodic) tasks

2. **Low Variance**: Bootstrap from estimates
   - Single-step noise vs full episode noise
   - Often faster convergence than MC

3. **Model-Free**: Learn from experience
   - No transition probabilities needed
   - Works in unknown environments

4. **Computationally Efficient**: Constant memory and time per step

### Bootstrapping Explained

**Bootstrapping** = using estimates to update estimates

**Methods that bootstrap**:
- ✅ Dynamic Programming: $V(s) = \mathbb{E}[r + \gamma V(s')]$
- ✅ TD Learning: $V(S_t) \leftarrow R_t + \gamma V(S_{t+1})$

**Methods that don't bootstrap**:
- ❌ Monte Carlo: $V(S_t) \leftarrow G_t$ (actual return)

**Trade-off**:
- Bootstrap → **Biased** (estimates may be wrong), **Lower variance**
- No bootstrap → **Unbiased**, **Higher variance**

## 2. TD(0) Prediction <a id='td-prediction'></a>

### Algorithm

**TD(0) Prediction** (one-step TD):

```
Input: policy π to evaluate
Initialize V(s) arbitrarily for all s ∈ S
Parameters: step size α ∈ (0, 1]

Repeat (for each episode):
  Initialize S
  
  Repeat (for each step of episode):
    A ← action given by π for S
    Take action A, observe R, S'
    V(S) ← V(S) + α[R + γV(S') - V(S)]
    S ← S'
  Until S is terminal
```

### Mathematical Foundation

**Bellman equation** for $V^\pi$:
$$V^\pi(s) = \mathbb{E}_\pi[R_{t+1} + \gamma V^\pi(S_{t+1}) | S_t = s]$$

**TD(0) uses sample to estimate expectation**:
$$V(S_t) \leftarrow V(S_t) + \alpha[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)]$$

**Interpretation**: 
- $R_{t+1} + \gamma V(S_{t+1})$ is a **sample** of the expected return
- TD(0) is **sample-based DP**

### Example: Random Walk

Consider a 1D random walk:
- States: A, B, C, D, E (+ two terminal states)
- Start in C
- Actions: Left or Right (50% each)
- Reward: 0 everywhere except +1 at right terminal

**True values** (exact): 
- $V(A) = 1/6$, $V(B) = 2/6$, $V(C) = 3/6$, $V(D) = 4/6$, $V(E) = 5/6$

**TD(0) learning**:
- Episode: C → D → E → Terminal (reward +1)
- After C→D transition (R=0):
  $$V(C) \leftarrow V(C) + \alpha[0 + \gamma V(D) - V(C)]$$
- After D→E transition (R=0):
  $$V(D) \leftarrow V(D) + \alpha[0 + \gamma V(E) - V(D)]$$
- After E→Terminal transition (R=1):
  $$V(E) \leftarrow V(E) + \alpha[1 + \gamma \cdot 0 - V(E)] = V(E) + \alpha[1 - V(E)]$$

Notice: **Updates happen immediately**, not at episode end!

### Terminal State Handling

For terminal states $S_T$:
$$V(S_T) = 0$$

So the final transition update is:
$$V(S_{T-1}) \leftarrow V(S_{T-1}) + \alpha[R_T + \gamma \cdot 0 - V(S_{T-1})]$$
$$= V(S_{T-1}) + \alpha[R_T - V(S_{T-1})]$$

## 3. TD vs MC vs DP <a id='comparison'></a>

### Comprehensive Comparison

| Aspect | DP | MC | TD |
|--------|----|----|----|
| **Model** | Required | Not required | Not required |
| **Bootstrapping** | Yes | No | Yes |
| **Sampling** | No (full distribution) | Yes | Yes |
| **Episodes** | Not required | Required | Not required |
| **Online/Offline** | Offline (full sweeps) | Offline (episode end) | **Online (each step)** |
| **Bias** | None | None | **Biased** (from bootstrap) |
| **Variance** | None (no sampling) | **High** | **Low** |
| **Convergence** | Guaranteed (finite MDP) | Guaranteed | Guaranteed (w/ conditions) |
| **Continuing tasks** | Yes | **No** | **Yes** |
| **Computational** | Expensive | Medium | **Cheap** |

### Backup Diagrams

**Dynamic Programming** (full backup):
```
    s
   /|\
  all actions
   /|\
  all next states (weighted by P)
```

**Monte Carlo** (sample trajectory):
```
s → s₁ → s₂ → ... → sT  (single trajectory to end)
```

**TD(0)** (sample one-step):
```
s → s' (single next state sample)
```

### Update Equations Comparison

**DP** (full expectation):
$$V(s) \leftarrow \sum_a \pi(a|s) \sum_{s',r} p(s',r|s,a)[r + \gamma V(s')]$$

**MC** (sample full return):
$$V(S_t) \leftarrow V(S_t) + \alpha[G_t - V(S_t)]$$
where $G_t = \sum_{k=0}^\infty \gamma^k R_{t+k+1}$

**TD(0)** (sample one-step return):
$$V(S_t) \leftarrow V(S_t) + \alpha[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)]$$

### Bias-Variance Analysis

**Monte Carlo Target**: $G_t$
- **Unbiased**: $\mathbb{E}[G_t] = V^\pi(S_t)$
- **High variance**: Stochastic over full trajectory

**TD(0) Target**: $R_{t+1} + \gamma V(S_{t+1})$
- **Biased**: $\mathbb{E}[R_{t+1} + \gamma V(S_{t+1})] \neq V^\pi(S_t)$ if $V \neq V^\pi$
- **Lower variance**: Stochastic over single step only

**Practical implications**:
- TD typically converges faster (lower variance)
- MC more robust to initialization (unbiased)
- TD sensitive to initial values (bias)

### Certainty Equivalence

**MC Advantage**: Converges to values that minimize MSE on training set

**TD Advantage**: Converges to **certainty-equivalence estimate**
- Maximum likelihood MDP model from data
- Solve that model exactly
- Often better generalization

**Example**: Limited data
- MC: May underfit (high variance)
- TD: Leverages Markov property for better estimates

## 4. SARSA: On-Policy TD Control <a id='sarsa'></a>

### From Prediction to Control

**Goal**: Find optimal policy using TD learning

**Approach**: Generalized Policy Iteration with TD
- Evaluation: Estimate $Q^\pi(s,a)$ using TD
- Improvement: Greedy policy w.r.t. $Q$

### SARSA Algorithm

**Name**: State-Action-Reward-State-Action (tuple used in update)

**Update rule**:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)]$$

**Algorithm**:
```
Initialize Q(s,a) arbitrarily, for all s ∈ S, a ∈ A
Parameters: step size α, exploration ε

Repeat (for each episode):
  Initialize S
  Choose A from S using policy derived from Q (ε-greedy)
  
  Repeat (for each step):
    Take action A, observe R, S'
    Choose A' from S' using policy derived from Q (ε-greedy)
    Q(S,A) ← Q(S,A) + α[R + γQ(S',A') - Q(S,A)]
    S ← S'; A ← A'
  Until S is terminal
```

### On-Policy Nature

SARSA is **on-policy** because:
- Uses $\epsilon$-greedy policy for both:
  - **Behavior** (generating actions)
  - **Evaluation** (updating Q-values)
- $A_{t+1}$ comes from the same policy being evaluated

### SARSA Convergence

**Theorem 4.1** (SARSA Convergence):

Under standard stochastic approximation conditions:
1. All state-action pairs visited infinitely often
2. Step-size: $\sum_t \alpha_t = \infty$, $\sum_t \alpha_t^2 < \infty$
3. Policy converges to greedy (GLIE)

Then $Q_t \to Q^*$ with probability 1.

### SARSA Tuple

Transition tuple: $(S_t, A_t, R_{t+1}, S_{t+1}, A_{t+1})$

**Mnemonic**: **S**-**A**-**R**-**S**-**A**

All five elements needed for update!

### Cliff Walking Example

Classic gridworld:
- Start at bottom-left
- Goal at bottom-right
- Cliff along bottom (reward -100, return to start)
- Normal steps: reward -1

**SARSA behavior**: 
- Learns **safe path** (away from cliff)
- Because $\epsilon$-greedy sometimes explores toward cliff
- Policy accounts for its own exploration

**This is on-policy**: Learns value of policy it's following (including exploration)

## 5. Q-Learning: Off-Policy TD Control <a id='q-learning'></a>

### The Breakthrough Algorithm

**Q-Learning** (Watkins, 1989) is one of the most important algorithms in RL.

**Key idea**: Learn optimal Q-values while following exploratory policy

### Q-Learning Update

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t)]$$

**Compare with SARSA**:
- SARSA: $R_{t+1} + \gamma Q(S_{t+1}, A_{t+1})$ (action actually taken)
- Q-Learning: $R_{t+1} + \gamma \max_a Q(S_{t+1}, a)$ (best action)

### Q-Learning Algorithm

```
Initialize Q(s,a) arbitrarily, for all s ∈ S, a ∈ A
Parameters: step size α, exploration ε

Repeat (for each episode):
  Initialize S
  
  Repeat (for each step):
    Choose A from S using ε-greedy policy from Q
    Take action A, observe R, S'
    Q(S,A) ← Q(S,A) + α[R + γ max_a Q(S',a) - Q(S,A)]
    S ← S'
  Until S is terminal
```

### Off-Policy Nature

Q-Learning is **off-policy** because:
- **Behavior policy**: $\epsilon$-greedy (exploratory)
- **Target policy**: Greedy (deterministic optimal)

**Behavior**: Choose $A_t$ using $\epsilon$-greedy

**Update**: Use $\max_a Q(S_{t+1}, a)$ (greedy)

### Q-Learning Convergence

**Theorem 5.1** (Q-Learning Convergence, Watkins & Dayan 1992):

Given:
1. All state-action pairs visited infinitely often
2. Step-sizes: $\sum_t \alpha_t(s,a) = \infty$, $\sum_t \alpha_t^2(s,a) < \infty$

Then $Q_t \to Q^*$ with probability 1.

**Note**: Does NOT require policy to converge to greedy! Can keep exploring forever.

### Cliff Walking Revisited

Same environment as SARSA.

**Q-Learning behavior**:
- Learns **optimal path** (close to cliff)
- Target policy is greedy (never explores)
- Behavior policy explores, but Q learns from optimal actions

**During learning**:
- Agent may fall off cliff (exploration)
- But Q-values converge to optimal policy

**Final greedy policy**: Hugs cliff (optimal)

### SARSA vs Q-Learning

| Aspect | SARSA | Q-Learning |
|--------|-------|------------|
| **Type** | On-policy | Off-policy |
| **Update** | $Q(S,A) + \alpha[R + \gamma Q(S',A') - Q(S,A)]$ | $Q(S,A) + \alpha[R + \gamma \max_a Q(S',a) - Q(S,A)]$ |
| **Learns** | Value of policy being followed | Value of optimal policy |
| **Exploration** | Affects learned values | Doesn't affect learned values |
| **Safety** | More conservative | More aggressive |
| **Convergence** | Requires GLIE | Works with any exploration |
| **Cliff Walking** | Safe path | Optimal path |

**When to use which**:
- **SARSA**: When exploration has cost (robots, real systems)
- **Q-Learning**: When can afford exploration (simulation, games)

## 6. Expected SARSA <a id='expected-sarsa'></a>

### The Hybrid Approach

**Expected SARSA** takes expectation over next actions instead of sampling.

### Update Rule

$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha\left[R_{t+1} + \gamma \sum_a \pi(a|S_{t+1}) Q(S_{t+1}, a) - Q(S_t, A_t)\right]$$

**Comparison**:
- **SARSA**: $R + \gamma Q(S', A')$ (sample next action)
- **Expected SARSA**: $R + \gamma \sum_a \pi(a|S') Q(S', a)$ (average over actions)
- **Q-Learning**: $R + \gamma \max_a Q(S', a)$ (special case when $\pi$ is greedy)

### Special Cases

**If $\pi$ is greedy**:
$$\sum_a \pi(a|s) Q(s,a) = Q(s, \arg\max_a Q(s,a)) = \max_a Q(s,a)$$

So Expected SARSA **= Q-Learning** when target policy is greedy!

### Advantages of Expected SARSA

1. **Lower variance** than SARSA (expectation vs sample)
2. **More stable** learning
3. **Generalizes both** SARSA and Q-Learning
4. **Can be on-policy or off-policy**

### Computational Cost

**SARSA/Q-Learning**: $O(1)$ per update

**Expected SARSA**: $O(|\mathcal{A}|)$ per update

Trade-off: Computation vs variance reduction

### Algorithm

```
Initialize Q(s,a) arbitrarily

Repeat (for each episode):
  Initialize S
  
  Repeat (for each step):
    Choose A from S using ε-greedy from Q
    Take action A, observe R, S'
    
    Expected_Q ← Σ_a π(a|S') Q(S',a)
    Q(S,A) ← Q(S,A) + α[R + γ Expected_Q - Q(S,A)]
    S ← S'
  Until S is terminal
```

## 7. Convergence Theory <a id='convergence'></a>

### Stochastic Approximation Theory

TD methods are **stochastic approximation** algorithms.

**General form**:
$$\theta_{t+1} = \theta_t + \alpha_t[Y_t - \theta_t]$$

where $Y_t$ is a noisy estimate of desired value.

### Robbins-Monro Conditions

**Theorem 7.1** (Robbins-Monro):

If step-sizes satisfy:
1. $\sum_{t=1}^\infty \alpha_t = \infty$ (steps large enough to overcome noise)
2. $\sum_{t=1}^\infty \alpha_t^2 < \infty$ (steps small enough to converge)

Then $\theta_t \to \theta^*$ almost surely.

**Examples**:
- $\alpha_t = \frac{1}{t}$ ✅ satisfies both
- $\alpha_t = \frac{1}{\sqrt{t}}$ ❌ violates first condition
- $\alpha_t = 0.01$ ❌ violates second condition

### TD(0) Convergence

**Theorem 7.2**: TD(0) converges to $V^\pi$ under:
1. Robbins-Monro step-size conditions
2. All states visited infinitely often
3. Bounded rewards

**Convergence rate**: $O(1/t)$ for table-based TD

### Q-Learning Convergence Proof Sketch

**Key insight**: Q-learning is a contraction mapping in expectation.

Define **Bellman optimality operator**:
$$(\mathcal{T}Q)(s,a) = \mathbb{E}[R + \gamma \max_{a'} Q(S',a')]$$

**Property**: $\mathcal{T}$ is a $\gamma$-contraction:
$$\|\mathcal{T}Q_1 - \mathcal{T}Q_2\|_\infty \leq \gamma \|Q_1 - Q_2\|_\infty$$

**Q-Learning**: Stochastic approximation to $\mathcal{T}$

$$Q_{t+1}(s,a) = Q_t(s,a) + \alpha_t[(\mathcal{T}Q_t)(s,a) + noise - Q_t(s,a)]$$

With Robbins-Monro conditions, noise averages out → convergence to unique fixed point $Q^*$.

### Convergence Speed Comparison

**Empirical observations** (problem-dependent):
- TD typically faster than MC (lower variance)
- Q-Learning can be slower than SARSA initially (off-policy)
- Expected SARSA often fastest (low variance)

**No universal winner**: Depends on:
- Problem structure
- Exploration strategy
- Step-size schedule

## 8. n-Step TD Methods <a id='n-step'></a>

### Bridging TD and MC

**Spectrum of methods**:
- $n=1$: TD(0) - one-step lookahead
- $n=\infty$: Monte Carlo - full episode
- $1 < n < \infty$: n-step methods - intermediate

### n-Step Return

**1-step return** (TD):
$$G_t^{(1)} = R_{t+1} + \gamma V(S_{t+1})$$

**2-step return**:
$$G_t^{(2)} = R_{t+1} + \gamma R_{t+2} + \gamma^2 V(S_{t+2})$$

**n-step return**:
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n V(S_{t+n})$$

**Full return** (MC):
$$G_t = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + ...$$

### n-Step TD Update

$$V(S_t) \leftarrow V(S_t) + \alpha[G_t^{(n)} - V(S_t)]$$

### n-Step SARSA

**n-step Q-return**:
$$G_t^{(n)} = R_{t+1} + \gamma R_{t+2} + ... + \gamma^{n-1} R_{t+n} + \gamma^n Q(S_{t+n}, A_{t+n})$$

**Update**:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[G_t^{(n)} - Q(S_t, A_t)]$$

### Choosing n

**Small n** (e.g., n=1):
- ✅ Low variance
- ✅ Fast updates
- ❌ High bias

**Large n** (approaching $\infty$):
- ✅ Low bias
- ❌ High variance
- ❌ Slow updates (wait n steps)

**Intermediate n** (e.g., n=3-5):
- Balance bias and variance
- Often best in practice

### Forward View vs Backward View

**Forward view**: Look ahead n steps for update
- Conceptually clear
- Requires waiting n steps

**Backward view**: Eligibility traces (Lesson 5)
- Update immediately
- Mathematically equivalent
- More efficient implementation

## 9. Maximization Bias <a id='max-bias'></a>

### The Problem

Q-Learning uses $\max_a Q(s,a)$ for updates.

**Issue**: $\mathbb{E}[\max_a Q(s,a)] \geq \max_a \mathbb{E}[Q(s,a)]$

**Result**: Systematic overestimation of values

### Example

State with 2 actions:
- True Q-values: $Q(s,a_1) = 0$, $Q(s,a_2) = 0$
- Estimates (noisy): $\hat{Q}(s,a_1) = -0.1$, $\hat{Q}(s,a_2) = +0.1$
- $\max_a \hat{Q}(s,a) = 0.1 > 0$ (overestimate!)

### Double Q-Learning Solution

**Idea**: Decouple action selection from action evaluation

**Maintain two Q-functions**: $Q_1$ and $Q_2$

**Update** (randomly choose which to update):

With probability 0.5:
$$Q_1(S,A) \leftarrow Q_1(S,A) + \alpha[R + \gamma Q_2(S', \arg\max_a Q_1(S',a)) - Q_1(S,A)]$$

With probability 0.5:
$$Q_2(S,A) \leftarrow Q_2(S,A) + \alpha[R + \gamma Q_1(S', \arg\max_a Q_2(S',a)) - Q_2(S,A)]$$

**Key**: Use $Q_1$ to select action, $Q_2$ to evaluate (and vice versa)

**Result**: Unbiased value estimates

### Double Q-Learning vs Q-Learning

**Q-Learning**: $\max_a Q(s,a)$ uses same Q for selection and evaluation

**Double Q-Learning**: Decouples selection ($Q_1$) and evaluation ($Q_2$)

**Cost**: 2x memory for two Q-tables

**Benefit**: Eliminates maximization bias

---

## Summary

In this notebook, you have learned:

1. ✅ **TD Learning** - Bootstrap + sample for online learning
2. ✅ **TD(0) Prediction** - One-step temporal difference
3. ✅ **TD vs MC vs DP** - Fundamental trade-offs
4. ✅ **SARSA** - On-policy TD control
5. ✅ **Q-Learning** - Off-policy TD control (breakthrough algorithm)
6. ✅ **Expected SARSA** - Hybrid approach
7. ✅ **Convergence Theory** - Stochastic approximation guarantees
8. ✅ **n-Step Methods** - Bridging TD and MC
9. ✅ **Maximization Bias** - Double Q-Learning solution

### Key Equations

**TD(0) Update**:
$$V(S_t) \leftarrow V(S_t) + \alpha[R_{t+1} + \gamma V(S_{t+1}) - V(S_t)]$$

**SARSA Update**:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[R_{t+1} + \gamma Q(S_{t+1}, A_{t+1}) - Q(S_t, A_t)]$$

**Q-Learning Update**:
$$Q(S_t, A_t) \leftarrow Q(S_t, A_t) + \alpha[R_{t+1} + \gamma \max_a Q(S_{t+1}, a) - Q(S_t, A_t)]$$

**TD Error**:
$$\delta_t = R_{t+1} + \gamma V(S_{t+1}) - V(S_t)$$

### Core Insights

1. **TD = DP + Sampling**: Best of both worlds
2. **Bootstrapping**: Lower variance, introduces bias
3. **Online learning**: Update every step, works for continuing tasks
4. **SARSA vs Q-Learning**: On-policy (safe) vs off-policy (optimal)
5. **Convergence**: Guaranteed under Robbins-Monro conditions

### Practical Guidance

**Use SARSA when**:
- Exploration has real cost
- Safety is important
- Online learning on physical systems

**Use Q-Learning when**:
- Learning in simulation
- Want optimal policy
- Can afford exploratory failures

**Use Expected SARSA when**:
- Want lower variance
- Can afford $O(|\mathcal{A}|)$ computation
- Generalizes both SARSA and Q-Learning

### Next Steps

**Lesson 4b** will implement:
- TD(0) prediction on random walk
- SARSA on GridWorld and CliffWorld
- Q-Learning comparison
- Expected SARSA
- Double Q-Learning
- Comprehensive performance analysis

---

**Lesson 4a Complete!** 🎉

You now understand the theoretical foundations of Temporal Difference learning, one of the most important concepts in reinforcement learning.